In this tutorial/guide, we will be exploring the MCP (Model Context Protocol) technique with Claude (this content is derived from Anthropic skilljar site)

MCP client serves as the communication bridge between your server and MCP servers. It is like an access point to all the tools that an MCP server provides. When you need to use external tools or services, the client handles all the message passing and protocol details for you.

One of MCP's key strengths is being transport agnostic which is a fancy way of saying the client and server can talk to each other using different communication methods.

The most common setup runs both the MCP client and server on the same machine where they communicate through standard input/output.

MCP clients and server can also connect over:
- HTTP
- WebSockers
- Various other network protocols

Once connected, the client and server exchange specific message types defined in the MCP specification. The main message types we will work with are:

- ListToolsRequest/ListToolsResult: client asks the server "what tools do you provide?" and gets back a list of available tools.

- CallToolRequest/CallToolResult: client asks the server to run a specific tool with certain arguments, then receives the results

Here's an example of how it will work in a real scenario:

1. A user asks "What repositories do I have?" and query is submitted to the server

2. Server realizes it needs to provide Claude with a list of available tools before making the request

3. Server asks the MCP client for tools which sends a `ListToolsRequest` to the MCP server and receives a `ListToolsResult` back

4. Now your server has everything needed to make the initial request to Claude - both the user's question and the available tools

5. Claude examines the tools and decides it needs to call one to answer the question. It responds with a tool use request.

6. Your server asks the MCP client to execute the tool Claude requested. The MCP client sends a `CallToolRequest` to the MCP server, which makes the actual request to GitHub

7. GitHub returns the repo data which flows back through the MCP server as a CallToolResult then to the MCP client, and finally to your server.

8. Your server sends the tool results back to Claude in a follow-up message. Claude now has all the info it needs to formulate a complete response

9. Finally Claude responds with the formatted answer, which your server passes back to the user.


Overall, this flow contains many steps but each component has a clear responsibility. The MCP client abstracts away the complexity of server communication, allowing us to focus on application logic.

Building an MCP server becomes much simpler when we use the official Python SDK. Instead of manually writing complex JSON schemas for tools, the SDK handles all the complexity with decorators and type hints.
In this example, we are creating an MCP server that manages documents stored in memory. The server will provide two essential tools:

1. Read document contents
2. Update them through find-and-replace operations



The following is to set up the MCP server

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")

For this implementation, documents are stored in a simple Python dictionary where keys are document IDs and values contain the document content:

```
# This is formatted as code
```



In [ ]:
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditure",
    "outlook.pdf": "This document presents the projected future performance of the",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment"
}

The SDK transforms tool creaton from a verbose process into something clean and readable using Python decorators and type hints.

The first tool allows Claude to read any document by its ID. Here's the implementation:

In [ ]:
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    return docs[doc_id]

The `@mcp.tool` decorator automatically generates the JSON schema that Claude needs. The `Field` class from Pydantic provides parameter descriptions that help Claude understand what each argument expects

The second tool performs a simple find-and-replace operations on documents. This takes three parameters: the document ID, the text to find, and the replacement text. The implementation uses Python's built-in string `replace()` method for simplicity

In [ ]:
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    docs[doc_id] = docs[doc_id].replace(old_str, new_str)

Both tools include basic error handling to manage cases where Claude requests a document that doesn't exist. This is what the `ValueError` is for which will return a descriptive message that Claude can understand and do something.